# Station Stacking - KAUS

Wide HRRR/GFS same-day 11am notebook for `KAUS`.

This workflow tunes XGBoost, LightGBM, and CatBoost with Optuna/TPE against RMSE on two fixed validation folds: train 2021-2023 validate 2024, train 2022-2024 validate 2025. It then trains on 2021-2025, tests on 2026, adds a Ridge meta-model stack, and simulates deterministic 2-degree weather brackets without calling Polymarket.


In [1]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

STATION_ID = "KAUS"
FAST_MODE = False
OPTUNA_TRIALS = 100
STACK_OPTUNA_TRIALS = 50
OPTUNA_VERBOSE = True
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
from src.calibration.station_stacking import (
    StationStackingConfig,
    missing_model_dependencies,
    run_station_year_split_experiment,
)


## Model Scores


In [3]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_verbose=OPTUNA_VERBOSE,
)
result = run_station_year_split_experiment(config)
result.scoreboard


[I 2026-06-06 09:00:33,093] A new study created in memory with name: no-name-5e6879a6-f12f-4883-b880-6d6b090196a5
[I 2026-06-06 09:00:38,481] Trial 0 finished with value: 6.536805233159389 and parameters: {'n_estimators': 799, 'learning_rate': 0.12369619597856178, 'max_depth': 6, 'min_child_weight': 2.385234757844707, 'gamma': 0.7800932022121826, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.5290418060840998, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.6677615511747083}. Best is trial 0 with value: 6.536805233159389.
[I 2026-06-06 09:01:57,762] Trial 1 finished with value: 6.299715442168637 and parameters: {'n_estimators': 1440, 'learning_rate': 0.0032515743808034223, 'max_depth': 8, 'min_child_weight': 8.23143373099555, 'gamma': 1.0616955533913808, 'subsample': 0.5909124836035503, 'colsample_bytree': 0.5917022549267169, 'reg_alpha': 5.472429642032198e-06, 'reg_lambda': 0.2922905212920093}. Best is trial 1 with value: 6.299715442168637.
[I 2026-06-06 09:02:33,413] Trial 2

,period,method,count,mae_f,rmse_f
0,validation_2024_2025,xgboost,596,4.190077,6.266381
1,validation_2024_2025,lightgbm,596,4.558466,6.724134
2,validation_2024_2025,catboost,596,4.225242,6.192952
3,validation_2024_2025,hrrr_raw,596,4.342980,6.152263
4,validation_2024_2025,gfs_raw,596,4.846561,7.070391
5,test_2026,xgboost,102,5.261474,6.365847
6,test_2026,lightgbm,102,5.333900,6.680975
7,test_2026,catboost,102,5.190067,6.391611
8,test_2026,ridge_stack,102,4.757982,5.916183
9,test_2026,hrrr_raw,102,4.018587,5.456511


## 2026 Weather Brackets


In [4]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bracket_accuracy_pct
0,xgboost,102,5.261474,6.365847,6.862745
1,lightgbm,102,5.333900,6.680975,13.72549
2,catboost,102,5.190067,6.391611,5.882353
3,ridge_stack,102,4.757982,5.916183,5.882353
4,hrrr_raw,102,4.018587,5.456511,11.764706
5,gfs_raw,102,3.971651,5.397913,14.705882


In [ ]:
import pandas as pd


def adjacent_brackets(bracket):
    if pd.isna(bracket):
        return []
    text = str(bracket).strip()
    if not text or "-" not in text:
        return []
    try:
        lower = int(text.split("-", 1)[0])
    except ValueError:
        return []
    return [
        f"{lower - 2}-{lower - 1}",
        f"{lower}-{lower + 1}",
        f"{lower + 2}-{lower + 3}",
    ]


bracket_3way = result.bracket_predictions.copy()

valid = bracket_3way["actual_bracket"].notna() & bracket_3way["predicted_bracket"].astype(str).str.strip().ne("")
bracket_3way = bracket_3way.loc[valid].copy()
bracket_3way["picked_brackets"] = bracket_3way["predicted_bracket"].map(adjacent_brackets)
bracket_3way["three_bracket_hit"] = bracket_3way.apply(
    lambda row: row["actual_bracket"] in row["picked_brackets"],
    axis=1,
)

three_bracket_accuracy = (
    bracket_3way
    .groupby("method", as_index=False)
    .agg(
        count=("three_bracket_hit", "size"),
        exact_bracket_accuracy_pct=("bracket_hit", lambda x: x.mean() * 100),
        three_bracket_accuracy_pct=("three_bracket_hit", lambda x: x.mean() * 100),
    )
    .sort_values("three_bracket_accuracy_pct", ascending=False)
)

three_bracket_accuracy
